In [1]:
import xarray as xr
import numpy as np
import glob
import re

In [3]:
data_dir = "/glade/derecho/scratch/ksha/EPRI_data/CESM2_SMYLE_OCN/"  # change this
paths = sorted(glob.glob(f"{data_dir}/SMYLE_????-11-01_daily_ensemble.zarr"))
# print(paths)

In [4]:
def process_forecast(path):
    # --- parse initialization year from filename, e.g. SST_1959.zarr ---
    m = re.search(r"SST_(\d{4})\.zarr", path)
    init_year = int(m.group(1)) if m else None

    # --- open Zarr ---
    ds = xr.open_zarr(path, chunks={"time": 100})  # tweak chunking if needed
    # ds: dims (time: 2981, nlat: 384, nlon: 320)
    # coords: time, TLAT(nlat,nlon), TLONG(nlat,nlon)
    sst = ds["SST"]

    # ------------------------------------------------------------------
    # (1) SST monthly climatology (for this 10‑year forecast only)
    # ------------------------------------------------------------------
    # Uses all time steps in this forecast, grouped by calendar month
    sst_clim = sst.groupby("time.month").mean("time")    # (month, nlat, nlon)

    # ------------------------------------------------------------------
    # (2) SST anomalies (daily) = SST - monthly climatology
    # ------------------------------------------------------------------
    sst_anom = sst.groupby("time.month") - sst_clim      # (time, nlat, nlon)

    # ------------------------------------------------------------------
    # (3) DJF Niño3.4 index from SST anomalies
    # ------------------------------------------------------------------
    lat = ds["TLAT"]           # (nlat, nlon)
    lon = ds["TLONG"]          # (nlat, nlon)

    # Decide whether longitude is 0–360 or -180–180
    if float(lon.max()) > 180:
        # assume 0–360
        lon_n34 = lon
        lon_min, lon_max = 190, 240    # 170W–120W => 190E–240E
    else:
        # assume -180–180
        lon_n34 = lon
        lon_min, lon_max = -170, -120  # 170W–120W

    # Niño3.4 box: 5S–5N, 170W–120W
    n34_mask = (
        (lat >= -5) & (lat <= 5) &
        (lon_n34 >= lon_min) & (lon_n34 <= lon_max)
    )

    # cos(lat) area weights within Niño3.4
    weights = np.cos(np.deg2rad(lat))
    weights = weights.where(n34_mask)

    # area‑weighted Niño3.4 SST anomaly time series (daily)
    # dims: time
    sst_n34 = (
        sst_anom.where(n34_mask)
                .weighted(weights)
                .mean(dim=("nlat", "nlon"))
    )

    # Convert to monthly mean Niño3.4 anomalies
    n34_mon = sst_n34.resample(time="MS").mean()

    # ---- DJF mean per year ----
    month = n34_mon["time"].dt.month
    year  = n34_mon["time"].dt.year

    # Define "DJF year": Dec of year N belongs to DJF of year N+1
    season_year = xr.where(month == 12, year + 1, year)

    # Keep only Dec–Jan–Feb and average over each DJF season
    n34_DJF = (
        n34_mon
        .where(month.isin([12, 1, 2]))
        .groupby(season_year)
        .mean("time")
    )
    # Rename dim from the default "group" to "year"
    n34_DJF = n34_DJF.rename({"group": "year"})

    # Attach some metadata
    n34_DJF.name = "nino34_DJF"
    n34_DJF.attrs.update({
        "long_name": "DJF mean Nino3.4 SST anomaly",
        "units": sst.attrs.get("units", "degC"),
        "init_year": init_year,
        "note": "SST anomalies relative to 10-yr monthly climatology of this forecast"
    })

    return init_year, sst_clim, sst_anom, n34_DJF

In [ ]:
results = {}

for p in paths:
    init_year, sst_clim, sst_anom, n34_DJF = process_forecast(p)
    results[init_year] = {
        "climatology": sst_clim,   # (month, nlat, nlon)
        "anom": sst_anom,          # (time, nlat, nlon)
        "nino34_DJF": n34_DJF      # (year)
    }

In [ ]:
init_year, sst_clim, sst_anom, n34_DJF = process_forecast(paths[0])